# Satria V2 — Augmentasi + Imbalance
4GB GPU | batch_size=4 | accumulation_steps=4 (efektif=16)

**Eksperimen:**
1. CLIP ViT-B/32: zero-shot + linear probe
2. EfficientNet-B0: RandAugment vs default aug
3. Focal Loss vs Weighted Sampler vs class_weights
4. MobileNetV3 + augmentasi terbaik

In [ ]:
import sys
sys.path.insert(0, "../..")

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.data.dataset import get_dataloaders, TrashDataset, get_transforms, _CLASS_TO_IDX
from modules.models.factory import TrashClassifier
from modules.models.clip_model import CLIPZeroShot, CLIPLinearProbe
from modules.training.train import fit
from modules.training.losses import FocalLoss
from modules.training.evaluate import compute_metrics
from modules.utils.load_data import load_train
from sklearn.model_selection import StratifiedShuffleSplit
import open_clip
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from torchvision import transforms

set_seed(SEED)
print(f"Device: {DEVICE}")

BATCH_SIZE = 4
ACCUMULATION_STEPS = 4
NUM_WORKERS = 2
EPOCHS_HEAD = 10
EPOCHS_FINETUNE = 15
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
PATIENCE = 8

results = []

---
## Experiment 1: CLIP ViT-B/32

### 1a. Zero-shot — langsung predict, 0 training

In [ ]:
df = load_train()
y = df["label"].map(_CLASS_TO_IDX).values
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(df, y))
df_train = df.iloc[train_idx].reset_index(drop=True)
df_val = df.iloc[val_idx].reset_index(drop=True)
print(f"Train: {len(df_train)} | Val: {len(df_val)}")

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)
print(f"CLIP model loaded, preprocess type: {type(clip_preprocess).__name__}")

In [ ]:
val_ds_clip = TrashDataset(df_val, transform=clip_preprocess)
val_loader_clip = DataLoader(
    val_ds_clip, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

In [ ]:
zs_model = CLIPZeroShot(device=DEVICE)
all_preds, all_labels = [], []
for images, labels in val_loader_clip:
    images = images.to(DEVICE)
    logits = zs_model(images)
    preds = logits.argmax(dim=1).cpu().numpy()
    all_preds.extend(preds.tolist())
    all_labels.extend(labels.cpu().numpy().tolist())

f1_macro, f1_per_class, prec, rec = compute_metrics(all_labels, all_preds)
print(f">>> CLIP Zero-shot: val_f1_macro = {f1_macro:.4f}")
print(f"    F1 per class: {[round(f, 4) for f in f1_per_class]}")
results.append({"name": "clip_zeroshot", "best_val_f1": f1_macro, "f1_per_class": f1_per_class})

### 1b. Linear probe — freeze encoder, train classifier

In [ ]:
clip_norm = clip_preprocess.transforms[-1]

train_clip_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    clip_norm,
])

train_ds_clip = TrashDataset(df_train, transform=train_clip_transform)
train_loader_clip = DataLoader(
    train_ds_clip, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)

In [ ]:
lp_model = CLIPLinearProbe(device=DEVICE)
res = fit(
    lp_model, train_loader_clip, val_loader_clip,
    name="satria_clip_linear_probe",
    accumulation_steps=ACCUMULATION_STEPS,
    epochs_head=EPOCHS_HEAD,
    epochs_finetune=0,
    lr_head=LR_HEAD,
    patience=PATIENCE,
    device=DEVICE,
    mode="linear_probe",
)
print(f">>> CLIP Linear Probe: val_f1_macro = {res['best_val_f1']:.4f}")
results.append(res)

---
## Experiment 2: EfficientNet-B0 — Augmentasi

### 2a. Default augmentation

In [ ]:
train_loader, val_loader, val_ds = get_dataloaders(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_randaugment=False,
)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

In [ ]:
model = TrashClassifier("efficientnet_b0", num_classes=NUM_CLASSES).to(DEVICE)
res = fit(
    model, train_loader, val_loader,
    name="satria_enb0_default_aug",
    encoder_name="efficientnet_b0",
    accumulation_steps=ACCUMULATION_STEPS,
    epochs_head=EPOCHS_HEAD,
    epochs_finetune=EPOCHS_FINETUNE,
    lr_head=LR_HEAD,
    lr_finetune=LR_FINETUNE,
    patience=PATIENCE,
    class_weights=val_ds.class_weights,
    device=DEVICE,
)
print(f">>> ENB0 Default Aug: val_f1_macro = {res['best_val_f1']:.4f}")
results.append(res)

### 2b. RandAugment

In [ ]:
train_loader_ra, val_loader_ra, val_ds_ra = get_dataloaders(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_randaugment=True,
)
print(f"Train: {len(train_loader_ra.dataset)} | Val: {len(val_loader_ra.dataset)}")

In [ ]:
model = TrashClassifier("efficientnet_b0", num_classes=NUM_CLASSES).to(DEVICE)
res = fit(
    model, train_loader_ra, val_loader_ra,
    name="satria_enb0_randaugment",
    encoder_name="efficientnet_b0",
    accumulation_steps=ACCUMULATION_STEPS,
    epochs_head=EPOCHS_HEAD,
    epochs_finetune=EPOCHS_FINETUNE,
    lr_head=LR_HEAD,
    lr_finetune=LR_FINETUNE,
    patience=PATIENCE,
    class_weights=val_ds_ra.class_weights,
    device=DEVICE,
)
print(f">>> ENB0 RandAugment: val_f1_macro = {res['best_val_f1']:.4f}")
results.append(res)

---
## Experiment 3: Imbalance Handling
All use RandAugment + EfficientNet-B0

### 3a. class_weights (baseline)
Sudah di-run di Experiment 2b — skor RandAugment di atas.

### 3b. Focal Loss (γ=2.0 + α=class_weights)

In [ ]:
model = TrashClassifier("efficientnet_b0", num_classes=NUM_CLASSES).to(DEVICE)
res = fit(
    model, train_loader_ra, val_loader_ra,
    name="satria_enb0_focal",
    encoder_name="efficientnet_b0",
    accumulation_steps=ACCUMULATION_STEPS,
    epochs_head=EPOCHS_HEAD,
    epochs_finetune=EPOCHS_FINETUNE,
    lr_head=LR_HEAD,
    lr_finetune=LR_FINETUNE,
    patience=PATIENCE,
    criterion=FocalLoss(gamma=2.0, alpha=val_ds_ra.class_weights.to(DEVICE)),
    device=DEVICE,
)
print(f">>> ENB0 Focal Loss: val_f1_macro = {res['best_val_f1']:.4f}")
results.append(res)

### 3c. Weighted Sampler

In [ ]:
train_loader_ws, val_loader_ws, val_ds_ws = get_dataloaders(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_randaugment=True,
    use_weighted_sampler=True,
)

In [ ]:
model = TrashClassifier("efficientnet_b0", num_classes=NUM_CLASSES).to(DEVICE)
res = fit(
    model, train_loader_ws, val_loader_ws,
    name="satria_enb0_weighted_sampler",
    encoder_name="efficientnet_b0",
    accumulation_steps=ACCUMULATION_STEPS,
    epochs_head=EPOCHS_HEAD,
    epochs_finetune=EPOCHS_FINETUNE,
    lr_head=LR_HEAD,
    lr_finetune=LR_FINETUNE,
    patience=PATIENCE,
    device=DEVICE,
)
print(f">>> ENB0 Weighted Sampler: val_f1_macro = {res['best_val_f1']:.4f}")
results.append(res)

---
## Experiment 4: MobileNetV3 + Best Config

In [ ]:
summary = pd.DataFrame(results)

aug_results = summary[summary["name"].isin([
    "satria_enb0_default_aug", "satria_enb0_randaugment"
])]
best_aug_name = aug_results.sort_values("best_val_f1", ascending=False).iloc[0]["name"]
use_randaug = "randaugment" in best_aug_name
print(f"Best aug: {best_aug_name} (F1={aug_results['best_val_f1'].max():.4f})")

loss_results = summary[summary["name"].isin([
    "satria_enb0_randaugment", "satria_enb0_focal", "satria_enb0_weighted_sampler"
])]
best_loss_name = loss_results.sort_values("best_val_f1", ascending=False).iloc[0]["name"]
print(f"Best loss config: {best_loss_name} (F1={loss_results['best_val_f1'].max():.4f})")

if "focal" in best_loss_name:
    print(f"  → Using Focal Loss")
    best_loss_fn = FocalLoss(gamma=2.0, alpha=val_ds_ra.class_weights.to(DEVICE))
    best_sampler = False
    best_cw = None
elif "weighted_sampler" in best_loss_name:
    print(f"  → Using WeightedSampler")
    best_loss_fn = None
    best_sampler = True
    best_cw = None
else:
    print(f"  → Using class_weights (baseline CE)")
    best_loss_fn = None
    best_sampler = False
    best_cw = None  # will be set from loader below

In [ ]:
if best_sampler:
    train_loader_best, val_loader_best, val_ds_best = get_dataloaders(
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        use_randaugment=use_randaug,
        use_weighted_sampler=True,
    )
    best_cw = None
else:
    train_loader_best, val_loader_best, val_ds_best = get_dataloaders(
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        use_randaugment=use_randaug,
    )
    best_cw = val_ds_best.class_weights

### 4a. MobileNetV3-Small

In [ ]:
model = TrashClassifier("mobilenetv3_small_100", num_classes=NUM_CLASSES).to(DEVICE)
res = fit(
    model, train_loader_best, val_loader_best,
    name="satria_mobilenetv3_small_best",
    encoder_name="mobilenetv3_small_100",
    accumulation_steps=ACCUMULATION_STEPS,
    epochs_head=EPOCHS_HEAD,
    epochs_finetune=EPOCHS_FINETUNE,
    lr_head=LR_HEAD,
    lr_finetune=LR_FINETUNE,
    patience=PATIENCE,
    class_weights=best_cw,
    criterion=best_loss_fn,
    device=DEVICE,
)
print(f">>> MNV3-Small Best: val_f1_macro = {res['best_val_f1']:.4f}")
results.append(res)

### 4b. MobileNetV3-Large

In [ ]:
model = TrashClassifier("mobilenetv3_large_100", num_classes=NUM_CLASSES).to(DEVICE)
res = fit(
    model, train_loader_best, val_loader_best,
    name="satria_mobilenetv3_large_best",
    encoder_name="mobilenetv3_large_100",
    accumulation_steps=ACCUMULATION_STEPS,
    epochs_head=EPOCHS_HEAD,
    epochs_finetune=EPOCHS_FINETUNE,
    lr_head=LR_HEAD,
    lr_finetune=LR_FINETUNE,
    patience=PATIENCE,
    class_weights=best_cw,
    criterion=best_loss_fn,
    device=DEVICE,
)
print(f">>> MNV3-Large Best: val_f1_macro = {res['best_val_f1']:.4f}")
results.append(res)

---
## Summary

In [ ]:
summary = pd.DataFrame(results)
summary = summary.sort_values("best_val_f1", ascending=False).reset_index(drop=True)
summary[["name", "best_val_f1", "f1_per_class"]].to_csv(RESULTS / "satria_v2_summary.csv", index=False)
summary[["name", "best_val_f1", "f1_per_class"]]